# 04 — Sparse Reconstruction: Lagrangian → Eulerian

**Unique benchmark task**: Reconstruct the full Eulerian velocity field from sparse Lagrangian particle observations.

This task leverages the **paired** Lagrangian–Eulerian data that makes CylinderWake3900 unique among fluid mechanics ML benchmarks. It is directly relevant to:
- **Data assimilation** (EnKF, variational methods, neural DA)
- **Physics-informed neural networks** (PINNs)
- **Sparse sensor reconstruction**

**Task**: Given N sparse particle positions + velocities, reconstruct the full 3D velocity field  
**Metric**: Relative L2 error between reconstructed and ground-truth Eulerian field  
**Challenge levels**: 100%, 10%, 1%, 0.1% particle density

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from cylinderwake import CylinderWake3900

%matplotlib inline
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Load Paired Data

In [ ]:
# Load both Eulerian (ground truth) and Lagrangian (sparse observations)
euler_ds = CylinderWake3900('eulerian', 'near', split='test')
lagr_ds  = CylinderWake3900('lagrangian', 'near', split='test')

print(f'Eulerian snapshots:  {len(euler_ds)}')
print(f'Lagrangian snapshots: {len(lagr_ds)}')

# The i-th snapshot in both datasets corresponds to the same physical time
euler_sample = euler_ds[0]
lagr_sample  = lagr_ds[0]

print(f"\nEulerian velocity shape: {euler_sample['velocity'].shape}")
print(f"Lagrangian positions:    {lagr_sample['positions'].shape}")
print(f"Lagrangian velocities:   {lagr_sample['velocities'].shape}")

## 2. Create Sparse Observation Masks

In [ ]:
def subsample_particles(positions, velocities, fraction=0.01):
    """Subsample particles to simulate sparse PTV observations."""
    n = positions.shape[0]
    n_keep = max(1, int(n * fraction))
    idx = np.random.choice(n, n_keep, replace=False)
    return positions[idx], velocities[idx]


pos = lagr_sample['positions'].numpy() if hasattr(lagr_sample['positions'], 'numpy') else lagr_sample['positions']
vel = lagr_sample['velocities'].numpy() if hasattr(lagr_sample['velocities'], 'numpy') else lagr_sample['velocities']

fractions = [1.0, 0.1, 0.01, 0.001]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, frac in zip(axes, fractions):
    sp, sv = subsample_particles(pos, vel, frac)
    speed = np.linalg.norm(sv, axis=1)
    ax.scatter(sp[:, 0], sp[:, 1], c=speed, s=5, alpha=0.6, cmap='viridis')
    ax.set_title(f'{frac*100:.1f}% → {sp.shape[0]} particles')
    ax.set_aspect('equal')
    ax.set_xlabel('x/D')

plt.suptitle('Sparse observation challenge levels', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Simple Nearest-Neighbor Baseline

A naive baseline: for each Eulerian grid point, assign the velocity of the nearest Lagrangian particle.

In [ ]:
from scipy.interpolate import griddata

def reconstruct_nearest_neighbor(sparse_pos, sparse_vel, grid_x, grid_y, grid_z, z_slice):
    """
    Reconstruct 2D velocity field at a z-slice using nearest-neighbor interpolation.
    """
    # Filter particles near the z-slice
    z_val = grid_z[z_slice]
    dz = np.abs(grid_z[1] - grid_z[0]) if len(grid_z) > 1 else 0.5
    mask = np.abs(sparse_pos[:, 2] - z_val) < dz
    
    if mask.sum() < 3:
        return None
    
    pos_2d = sparse_pos[mask, :2]
    vel_2d = sparse_vel[mask]
    
    # Create meshgrid
    X, Y = np.meshgrid(grid_x, grid_y, indexing='ij')
    points = np.column_stack([X.ravel(), Y.ravel()])
    
    # Interpolate each component
    recon = np.zeros((len(grid_x), len(grid_y), 3))
    for i in range(3):
        recon[:, :, i] = griddata(
            pos_2d, vel_2d[:, i], (X, Y), method='nearest'
        )
    
    return recon


print('Nearest-neighbor reconstruction baseline ready.')
print('For proper benchmarking, implement your method and compare against the Eulerian ground truth.')
print('\nSuggested approaches:')
print('  1. Physics-Informed Neural Networks (PINNs)')
print('  2. Neural Radiance Fields adapted for flow (NeRF-Flow)')
print('  3. Graph Neural Networks on particle positions')
print('  4. Ensemble Kalman Filter (EnKF)')

## Benchmark Results Table

| Method | 100% particles | 10% | 1% | 0.1% |
|--------|:-----------:|:---:|:--:|:----:|
| Nearest-neighbor | TBD | TBD | TBD | TBD |
| RBF interpolation | — | — | — | — |
| PINN | — | — | — | — |
| GNN | — | — | — | — |

**We welcome contributions!** Submit your results via Pull Request.